<a href="https://colab.research.google.com/github/rhodes-byu/stat-486/blob/main/notebooks/08-mlps-keras.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a><p><b></b></p>

# CIFAR-10 Neural Networks Tutorial: scikit-learn MLP vs Keras Feed-Forward Network

This notebook has two parts:

1. **Part 1**: Use `MLPClassifier` in a scikit-learn pipeline for CIFAR-10 image classification.
2. **Part 2**: Build a feed-forward neural network from scratch in Keras (with a callback) and compare performance.

> CIFAR-10 has 60,000 color images (32×32×3) across 10 classes. The task is supervised multiclass classification.

## Part 1 — CIFAR-10 with scikit-learn `MLPClassifier`

### Reading map (Part 1)
1. **Cell 3**: review imports and the two libraries used in this notebook (`scikit-learn` and `Keras`).
2. **Cell 4**: inspect CIFAR-10 shape, labels, and sample images.
3. **Cells 5–6**: follow flattening + pipeline training, then read test performance.

### Dataset characteristics
- **Input**: 32×32 RGB images (3 channels)
- **Classes**: 10 labels (`airplane`, `automobile`, `bird`, `cat`, `deer`, `dog`, `frog`, `horse`, `ship`, `truck`)
- **Split**: 50,000 train and 10,000 test images
- **Problem type**: multiclass image classification

### API questions students often ask (scikit-learn)
- **Why use `Pipeline`?** It guarantees that preprocessing (`StandardScaler`) is applied the same way during both training and prediction.
- **Why call `fit` only once on the pipeline?** `pipeline.fit(...)` trains each step in order; you do not separately train the scaler and classifier.
- **Why `predict` on test data instead of probabilities?** `predict` returns class labels directly; `predict_proba` would return class probabilities.
- **Why flatten images first?** `MLPClassifier` in scikit-learn expects tabular input of shape `(n_samples, n_features)`.

In this first section, we flatten each image into a feature vector and train an `MLPClassifier` inside a scikit-learn pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import joblib

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
tf.random.set_seed(42)

In [ ]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()
y_train = y_train.ravel()
y_test = y_test.ravel()

class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print("Train shape:", x_train.shape, " Test shape:", x_test.shape)
print("Number of classes:", len(class_names))
print("Sample labels:", [class_names[i] for i in y_train[:10]])

# Quick visual check: 8 sample images
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for idx, ax in enumerate(axes.ravel()):
    ax.imshow(x_train[idx])
    ax.set_title(class_names[y_train[idx]])
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Flatten images for scikit-learn (n_samples, 3072)
X_train_flat = x_train.reshape(x_train.shape[0], -1).astype("float32")
X_test_flat = x_test.reshape(x_test.shape[0], -1).astype("float32")

# Optional subsample for faster training in class/demo settings
train_subset = 15000
X_train_sub = X_train_flat[:train_subset]
y_train_sub = y_train[:train_subset]

print("Flattened train subset:", X_train_sub.shape)
print("Flattened test:", X_test_flat.shape)

In [ ]:
# MLPClassifier pipeline
# Pipeline order matters: output of scaler becomes input to MLPClassifier.
sk_mlp_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    (
        "mlp",
        MLPClassifier(
            hidden_layer_sizes=(256,),
            activation="relu",
            solver="adam",
            learning_rate_init=1e-3,
            max_iter=30,
            batch_size=256,
            early_stopping=True,
            n_iter_no_change=5,
            random_state=42,
            verbose=False,
        ),
    ),
])

# fit() trains all steps in the pipeline in sequence.
sk_mlp_pipeline.fit(X_train_sub, y_train_sub)

# predict() returns class labels for each test example.
y_pred_sk = sk_mlp_pipeline.predict(X_test_flat)

acc_sk = accuracy_score(y_test, y_pred_sk)
print(f"scikit-learn MLP test accuracy: {acc_sk:.4f}")
print("\nClassification report (scikit-learn MLP):")
print(classification_report(y_test, y_pred_sk, target_names=class_names, digits=3))

### Glossary 1 — Core neural-network terms

- **Tensor**: a multi-dimensional array of numbers used to store data.
- **Feature**: one input variable to the model (here, one pixel-channel value after flattening).
- **Parameter**: a learned value (weights and biases) updated during training.
- **Activation function**: nonlinearity applied after a layer's weighted sum (e.g., ReLU).
- **Loss**: numerical measure of prediction error that training tries to minimize.
- **Optimizer**: algorithm that updates parameters to reduce loss (e.g., Adam).
- **Batch**: a small subset of training examples processed in one optimization step.
- **Epoch**: one full pass through the training dataset.
- **Overfitting**: when training performance improves but test/validation performance does not.

## Part 2 — Feed-Forward Neural Network with Keras

### Reading map (3 quick steps)
1. **Cell 9**: follow how CIFAR-10 images are prepared (flattened, normalized, batched) for Keras.
2. **Cell 10**: focus on how the model is constructed (`Sequential`, `Dense`, `Dropout`) and compiled.
3. **Cells 10–11**: observe training with `EarlyStopping`, then compare final test accuracy to Part 1.

### Glossary 2 — Keras-specific terms
- **`keras.Sequential`**: stacks layers in a simple top-to-bottom order.
- **`layers.Input(shape=...)`**: defines the expected shape of one input example.
- **`layers.Dense(units, activation=...)`**: fully connected layer with a chosen activation.
- **`layers.Dropout(rate)`**: randomly drops units during training for regularization.
- **`model.compile(...)`**: sets optimizer, loss, and metrics before training.
- **`model.fit(...)`**: runs training for some number of epochs.
- **`model.evaluate(...)`**: computes loss/metrics on held-out data.
- **`callbacks`**: hooks that modify behavior during training (e.g., `EarlyStopping`).
- **`tf.data.Dataset`**: efficient input pipeline for batching/shuffling/prefetching.

### API questions students often ask (Keras)
- **Why do we call `compile` before `fit`?** Keras needs to know the loss function, optimizer, and metrics before it can run training.
- **Why does the output layer have 10 units?** CIFAR-10 has 10 classes, so we output one score/probability per class.
- **Where does `Input(shape=(3072,))` come from?** Each 32×32 RGB image has $32\times32\times3=3072$ values after flattening into a 1D vector.
- **Why `softmax` with `categorical_crossentropy`?** This is the standard multiclass pairing when labels are one-hot encoded.
- **Could we keep integer labels instead of one-hot encoding?** Yes—then use `sparse_categorical_crossentropy` instead.
- **How can we inspect model architecture and parameter counts?** Use `model.summary()` after building the model.
- **What does `validation_data` do in `fit`?** It evaluates the model each epoch on held-out data to monitor generalization.
- **What does `EarlyStopping` monitor?** Here it monitors `val_loss` and stops training when it stops improving.

In this part, we build a neural network using Keras' high-level API.

A basic feed-forward neural network has these key components:
- **Input layer**: receives features (here, 3072 values from 32×32×3 images after flattening)
- **Hidden Dense layers**: each neuron computes a weighted sum plus bias, then applies an activation (ReLU)
- **Dropout layers**: randomly turn off some units during training to reduce overfitting
- **Output layer**: 10-unit softmax for class probabilities
- **Loss + optimizer**: `categorical_crossentropy` with Adam for multiclass training
- **Callback**: training control, e.g., early stopping

### How Keras constructs the model
1. **Define architecture** with `keras.Sequential([...])`.
2. **Compile** with an optimizer, loss, and metrics (`model.compile(...)`).
3. **Train** with `model.fit(...)` using training and validation data.
4. **Evaluate** generalization with `model.evaluate(...)` on test data.

For first exposure, this workflow is a practical mental model: **define → compile → fit → evaluate**.

In [ ]:
# For Dense (fully connected) networks, each image must be a 1D vector.
# 32 * 32 * 3 = 3072 features per image.
# We also scale pixel values from [0, 255] to [0, 1] to help optimization.
X_train_nn = x_train.reshape(x_train.shape[0], -1).astype("float32") / 255.0
X_test_nn = x_test.reshape(x_test.shape[0], -1).astype("float32") / 255.0

# Use the same subset size as the sklearn example for a fair runtime comparison.
y_train_nn = y_train.copy()
y_test_nn = y_test.copy()

X_train_nn_sub = X_train_nn[:train_subset]
y_train_nn_sub = y_train_nn[:train_subset]

# Softmax + categorical_crossentropy expects one-hot labels like [0,0,1,0,...].
y_train_nn_sub_oh = keras.utils.to_categorical(y_train_nn_sub, num_classes=10)
y_test_nn_oh = keras.utils.to_categorical(y_test_nn, num_classes=10)

# tf.data pipeline:
# - from_tensor_slices creates (x, y) examples
# - shuffle randomizes order each epoch
# - batch groups examples for vectorized training
# - prefetch overlaps input work with model execution
batch_size = 128
train_ds = (
    tf.data.Dataset.from_tensor_slices((X_train_nn_sub, y_train_nn_sub_oh))
    .shuffle(buffer_size=5000, seed=42)
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

test_ds = (
    tf.data.Dataset.from_tensor_slices((X_test_nn, y_test_nn_oh))
    .batch(batch_size)
    .prefetch(tf.data.AUTOTUNE)
)

print("Keras train subset:", X_train_nn_sub.shape)
print("Keras test set:", X_test_nn.shape)
print("Feature count per image:", X_train_nn_sub.shape[1])

### Why these preprocessing choices?

- **Flattening** converts each image to a vector so it can be used by Dense layers.
- **Normalization** keeps feature scales small and consistent, which usually stabilizes gradient-based learning.
- **Mini-batches** (`batch_size=128`) provide a balance between noisy single-sample updates and expensive full-dataset updates.
- **One-hot labels** match the softmax output and categorical cross-entropy loss.

In later courses, convolutional layers avoid flattening early and usually perform better on image structure, but this dense setup is ideal for first principles.

In [ ]:
# Build a feed-forward network from scratch using Sequential.
# Layer order is exactly the order data flows through the model.
# Why input shape is 3072:
# - CIFAR-10 image shape is (32, 32, 3)
# - Flattening gives 32 * 32 * 3 = 3072 features per image
keras_ffn = keras.Sequential([
    layers.Input(shape=(3072,)),            # One flattened image vector
    layers.Dense(512, activation="relu"),  # Hidden layer 1
    layers.Dropout(0.3),                    # Regularization
    layers.Dense(256, activation="relu"),  # Hidden layer 2
    layers.Dropout(0.3),                    # Regularization
    layers.Dense(10, activation="softmax"), # 10 class probabilities
])

# summary() is a quick API check for layer shapes and parameter counts.
keras_ffn.summary()

# Compile chooses HOW the model learns:
# - optimizer: update rule for weights
# - loss: objective to minimize
# - metrics: what to report during training/evaluation
keras_ffn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

# Callback example: EarlyStopping
# If validation loss does not improve for 3 epochs, training stops.
# restore_best_weights=True rolls back to the best validation-loss epoch.
early_stopping_cb = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
)

# fit() performs the training loop over epochs.
# validation_data allows monitoring generalization each epoch.
# The returned History object stores epoch-by-epoch metrics in history.history.
history = keras_ffn.fit(
    train_ds,
    epochs=20,
    validation_data=test_ds,
    callbacks=[early_stopping_cb],
    verbose=1,
)
print("History keys:", list(history.history.keys()))

# evaluate() runs the model on held-out test data.
test_loss_keras, test_acc_keras = keras_ffn.evaluate(test_ds, verbose=0)
print(f"Keras feed-forward test accuracy: {test_acc_keras:.4f}")

### Training history (what `fit()` recorded)

This plot uses the `history` object returned by `model.fit(...)`.

- **Training curves** (`loss`, `accuracy`) show performance on the training batches.
- **Validation curves** (`val_loss`, `val_accuracy`) show performance on held-out data after each epoch.
- When validation loss stops improving, `EarlyStopping` may end training early.

In [ ]:
# Plot training vs validation history
history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss curves
axes[0].plot(history_df.index + 1, history_df["loss"], label="train_loss")
if "val_loss" in history_df.columns:
    axes[0].plot(history_df.index + 1, history_df["val_loss"], label="val_loss")
axes[0].set_title("Loss by epoch")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

# Accuracy curves
axes[1].plot(history_df.index + 1, history_df["accuracy"], label="train_accuracy")
if "val_accuracy" in history_df.columns:
    axes[1].plot(history_df.index + 1, history_df["val_accuracy"], label="val_accuracy")
axes[1].set_title("Accuracy by epoch")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

### Saving and loading trained models

After training, we often want to persist models for reuse later (without retraining).

- **scikit-learn pipeline**: save with `joblib.dump(...)`, load with `joblib.load(...)`.
- **Keras model**: save with `model.save(...)`, load with `keras.models.load_model(...)`.

We use a `saved_models` folder and then run a quick reload check by predicting/evaluating with the loaded models.

In [ ]:
# Create an output folder for persisted models
save_dir = Path("saved_models")
save_dir.mkdir(exist_ok=True)

# -----------------------------
# 1) Save and load sklearn pipeline
# -----------------------------
sk_model_path = save_dir / "cifar10_sklearn_mlp_pipeline.joblib"
joblib.dump(sk_mlp_pipeline, sk_model_path)

loaded_sk_model = joblib.load(sk_model_path)
loaded_sk_pred = loaded_sk_model.predict(X_test_flat)
loaded_sk_acc = accuracy_score(y_test, loaded_sk_pred)

print(f"Saved sklearn pipeline to: {sk_model_path}")
print(f"Reloaded sklearn test accuracy: {loaded_sk_acc:.4f}")

# -----------------------------
# 2) Save and load Keras model
# -----------------------------
keras_model_path = save_dir / "cifar10_keras_ffn.keras"
keras_ffn.save(keras_model_path)

loaded_keras_model = keras.models.load_model(keras_model_path)
loaded_keras_loss, loaded_keras_acc = loaded_keras_model.evaluate(test_ds, verbose=0)

print(f"Saved Keras model to: {keras_model_path}")
print(f"Reloaded Keras test accuracy: {loaded_keras_acc:.4f}")

# Example inference question students ask:
# "How do I turn model outputs into a class label?"
sample_probs = loaded_keras_model.predict(X_test_nn[:1], verbose=0)
sample_pred_idx = int(np.argmax(sample_probs[0]))
print("Example predicted class:", class_names[sample_pred_idx])
print("Top probability:", float(np.max(sample_probs[0])))

In [ ]:
# Compare final performance
results = pd.DataFrame(
    {
        "Model": ["scikit-learn MLPClassifier", "Keras Feed-Forward NN"],
        "Test Accuracy": [acc_sk, test_acc_keras],
    }
)

results = results.sort_values("Test Accuracy", ascending=False).reset_index(drop=True)
print(results)

winner = results.loc[0, "Model"]
margin = results.loc[0, "Test Accuracy"] - results.loc[1, "Test Accuracy"]
print(f"\nTop model: {winner}")
print(f"Accuracy margin: {margin:.4f}")

### Notes on expected outcomes

- The Keras model often performs better because it is easier to scale depth/regularization and optimize with mini-batches on modern hardware.
- The scikit-learn MLP pipeline is still a useful baseline and demonstrates how neural-net-style classifiers fit into the sklearn workflow.
- CIFAR-10 is a natural image task where **convolutional networks** typically outperform pure feed-forward models, which can be explored next.